# Tutorial RAG con LangChain + Ollama (TinyLlama)

En este notebook construirás un sistema **RAG (Retrieval-Augmented Generation)** paso a paso usando:

- **LLM de respuesta**: `tinyllama` en Ollama
- **Embeddings**: `nomic-embed-text` en Ollama
- **Orquestación**: LangChain
- **Vector store**: Chroma

---

## Qué lograrás

1. Cargar documentos de texto
2. Dividirlos en chunks
3. Indexarlos en una base vectorial
4. Recuperar contexto relevante
5. Responder preguntas usando TinyLlama con contexto recuperado

## 1) Instalación de librerías

Ejecuta esta celda una sola vez. Si ya las tienes, puedes omitirla.

In [1]:
%pip install -qU langchain langchain-community langchain-ollama chromadb pypdf

Note: you may need to restart the kernel to use updated packages.


## 2) Verificar Ollama y descargar modelos

Asegúrate de tener corriendo Ollama localmente (`http://localhost:11434`).

Modelos usados en este tutorial:

- `tinyllama` (generación)
- `nomic-embed-text` (embeddings)

In [ ]:
import subprocess

for model in ["tinyllama", "nomic-embed-text"]:
    print(f"Pulling {model}...")
    subprocess.run(["ollama", "pull", model], check=True)

print("Modelos listos.")


Pulling tinyllama...
Pulling nomic-embed-text...
Modelos listos.


## 3) Crear un pequeño corpus de documentos

Para el tutorial usaremos textos cortos locales. Luego puedes reemplazarlos por tus PDFs, DOCX o páginas web.

In [3]:
from pathlib import Path

data_dir = Path("./rag_docs")
data_dir.mkdir(parents=True, exist_ok=True)

docs = {
    "economia.txt": "La inflación es el aumento sostenido de los precios en una economía.",
    "ia.txt": "El aprendizaje automático permite a los sistemas aprender patrones desde datos.",
    "energia.txt": "La energía solar convierte radiación del sol en electricidad mediante paneles fotovoltaicos.",
}

for filename, content in docs.items():
    (data_dir / filename).write_text(content, encoding="utf-8")

print("Documentos creados en:", data_dir.resolve())

Documentos creados en: E:\CODES\llms\TALLER_LLM\PARTE_III\rag_docs


## 4) Cargar, dividir e indexar documentos en Chroma

In [4]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

loader = DirectoryLoader(
    str(data_dir),
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
 )
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(documents)

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="tutorial_rag_tinyllama"
)

print(f"Documentos: {len(documents)} | Chunks: {len(chunks)}")

e:\OPT\MINICONDA\envs\llms-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Documentos: 3 | Chunks: 3


## 5) Construir cadena RAG con TinyLlama (Ollama)

In [6]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatOllama(
    model="tinyllama",
    temperature=0.2
 )

prompt = ChatPromptTemplate.from_template(
    """Responde SOLO con base en el contexto. Si no está en el contexto, di: 'No está en los documentos'.

Contexto:
{context}

Pregunta: {question}"""
 )

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": lambda x: x}
    | prompt
    | llm
    | StrOutputParser()
)

## 6) Probar consultas con RAG

In [7]:
questions = [
    "¿Qué es la inflación?",
    "¿Cómo funciona la energía solar?",
    "¿Qué permite el aprendizaje automático?",
    "¿Qué opinan los documentos sobre medicina cuántica?"
]

for q in questions:
    print(f"\nPregunta: {q}")
    print("Respuesta:", rag_chain.invoke(q))


Pregunta: ¿Qué es la inflación?
Respuesta: Respuesta SOLO con base en el contexto. Si no está en los documentos, di: 'No está en los documentos'.

Concepto:
La inflación es el aumento de los precios en una economía.

Escuela:
El aprendizaje automático permite a los sistemas aprender patrones desde datos.

Energía solar convertir radiaciones del sol en electricidad mediante paneles fotovoltaicos.

Pregunta: ¿Cómo funciona la energía solar?
Respuesta: Respuesta SOLO con base en el contexto:

No está en los documentos.

Pregunta: ¿Qué permite el aprendizaje automático?
Respuesta: Respuesta SOLO con base en el contexto. Si no está en los documentos, di: 'No está en los documentos'.

Convierte radiación del sol en electricidad mediante paneles fotovoletajicos.

Pregunta: ¿Qué opinan los documentos sobre medicina cuántica?
Respuesta: Respuesta SOLO con base en el contexto. Si no está en el contexto, di: 'No está en los documentos'.

Contexto:
La inflación es el aumento sosteniendo de los pr

## 7) Inspeccionar qué recupera el retriever

In [8]:
query = "¿Qué es la inflación?"
retrieved = retriever.invoke(query)

print("Consulta:", query)
print("\nChunks recuperados:\n")
for i, doc in enumerate(retrieved, start=1):
    print(f"[{i}]", doc.page_content)

Consulta: ¿Qué es la inflación?

Chunks recuperados:

[1] La inflación es el aumento sostenido de los precios en una economía.
[2] El aprendizaje automático permite a los sistemas aprender patrones desde datos.
[3] La energía solar convierte radiación del sol en electricidad mediante paneles fotovoltaicos.


## 8) Siguientes pasos (producción)

- Cambia `rag_docs/*.txt` por tus documentos reales (`PDF`, `DOCX`, `HTML`).
- Ajusta `chunk_size` y `chunk_overlap` según tamaño de documentos.
- Usa un modelo más fuerte para respuesta final si necesitas más calidad (por ejemplo `llama3.1:8b`).
- Mantén embeddings dedicados (ej. `nomic-embed-text`) para recuperación estable.

## Orden de ejecución recomendado

1. Instalación
2. Pull de modelos en Ollama
3. Crear corpus
4. Indexar en Chroma
5. Construir cadena RAG
6. Probar preguntas
7. Inspeccionar chunks recuperados